# NER mit gbert-base auf dem GPTNERMED-Datensatz

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q \
  "transformers>=4.40,<4.46" \
  "datasets>=2.16.0,<3.0.0" \
  "accelerate>=0.30" \
  "seqeval" \
  "evaluate>=0.4.0,<0.5.0"

In [ ]:
import torch
import datasets
import transformers

print(torch.cuda.is_available())

## Daten laden

In [ ]:
import datasets

# Lädt Train/Validation/Test direkt vom Hugging Face Hub
raw = datasets.load_dataset("jfrei/GPTNERMED", trust_remote_code=True)

# ner_class ist ein ClassLabel (also ints) -> zurück auf die Namen mappen
class_names = raw["train"].features["ner_labels"].feature["ner_class"].names  # ["Medikation", "Dosis", "Diagnose"]

def to_offset_format(example):
    labels = example["ner_labels"]  # dict mit Listen: start / stop / ner_class
    return {
        "text": example["sentence"],
        "ent_starts": labels["start"],
        "ent_stops": labels["stop"],
        "ent_classes": [class_names[c] for c in labels["ner_class"]],
    }

dataset = raw.map(to_offset_format, remove_columns=["sentence", "ner_labels"])

print(f"Loaded {len(dataset['train'])} train, {len(dataset['validation'])} dev and {len(dataset['test'])} test")
print(dataset["train"][2])

## Daten vorverarbeiten

BIO schema: 
- B: Anfang der Entität
- I: Innerhalb der Entität
- O: keine Entität


In [ ]:
from transformers import AutoTokenizer

label_list = ["O", "B-Medikation", "I-Medikation", "B-Dosis", "I-Dosis", "B-Diagnose", "I-Diagnose"]
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

tokenizer = AutoTokenizer.from_pretrained("deepset/gbert-base")

In [ ]:
def preprocess_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, max_length=128, return_offsets_mapping=True)

    all_labels = []
    for i, offsets in enumerate(tokenized["offset_mapping"]):
        labels = []
        for start, end in offsets:
            if start == end:
                labels.append(-100)
                continue
            token_label = "O"
            for j in range(len(examples["ent_starts"][i])):
                if start >= examples["ent_starts"][i][j] and end <= examples["ent_stops"][i][j]:
                    if start == examples["ent_starts"][i][j]:
                        token_label = "B-" + examples["ent_classes"][i][j]
                    else:
                        token_label = "I-" + examples["ent_classes"][i][j]
            labels.append(label2id[token_label])
        all_labels.append(labels)

    tokenized["labels"] = all_labels
    tokenized.pop("offset_mapping")
    return tokenized

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=["text", "ent_starts", "ent_stops", "ent_classes"])
tokenized_dataset = tokenized_dataset.shuffle(seed=42)

In [ ]:
example = tokenized_dataset["train"][0]
tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
for token, label in zip(tokens, example["labels"]):
    print(token, "->", id2label[label] if label != -100 else "-100")

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for pred, lab in zip(predictions, labels):
        sentence_preds = []
        sentence_labels = []
        for p, l in zip(pred, lab):
            if l != -100:
                sentence_preds.append(label_list[p])
                sentence_labels.append(label_list[l])
        true_predictions.append(sentence_preds)
        true_labels.append(sentence_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    metrics = {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }
    
    for entity in ["Medikation", "Dosis", "Diagnose"]:
        if entity in results:
            metrics[f"{entity}_f1"] = results[entity]["f1"]
    return metrics

## Training (über drei Seeds)

In [ ]:
import wandb
wandb.init(mode="disabled")

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, set_seed

seeds = [42, 123, 2024]
seed_results = []  # sammelt die Test-Metriken pro Seed

for seed in seeds:
    print(f"\n===== Training mit Seed {seed} =====")
    set_seed(seed)  # macht u.a. die Gewichts-Init des Klassifikations-Kopfes reproduzierbar

    model = AutoModelForTokenClassification.from_pretrained(
        "deepset/gbert-base", num_labels=7, id2label=id2label, label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir=f"gbert_gptnermed_seed{seed}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Test-Metriken für diesen Seed sichern
    test_metrics = trainer.predict(tokenized_dataset["test"]).metrics
    test_metrics["seed"] = seed
    seed_results.append(test_metrics)
    print(f"Seed {seed}: Test-F1 = {test_metrics['test_f1']:.4f}")

    # Bestes Modell dieses Seeds abspeichern (durch load_best_model_at_end=True ist
    # trainer.model bereits das beste Modell aus dem Training). Landet im Colab-Dateisystem.
    save_dir = f"/content/drive/MyDrive/gbert-base-no-hp-seed{seed}"
    trainer.save_model(save_dir)          # speichert Modell-Gewichte + Config
    tokenizer.save_pretrained(save_dir)   # speichert den Tokenizer dazu
    print(f"Seed {seed}: bestes Modell gespeichert unter '{save_dir}'")

print("\nAlle Seeds trainiert!")

# Letztes trainiertes Modell für Test-Auswertung und Inferenz behalten
model = trainer.model.module if hasattr(trainer.model, 'module') else trainer.model

In [ ]:
import numpy as np

entities = ["Medikation", "Dosis", "Diagnose"]

print("Test-Ergebnisse pro Seed:")
for r in seed_results:
    per_entity = "  ".join(f"{e}-F1={r[f'test_{e}_f1']:.4f}" for e in entities)
    print(f"  Seed {r['seed']}: F1={r['test_f1']:.4f}  Precision={r['test_precision']:.4f}  Recall={r['test_recall']:.4f}  |  {per_entity}")

f1_scores = [r["test_f1"] for r in seed_results]
prec_scores = [r["test_precision"] for r in seed_results]
rec_scores = [r["test_recall"] for r in seed_results]

print("\nÜber alle Seeds gemittelt (Mittelwert ± Standardabweichung):")
print(f"  F1 (gesamt): {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  Precision:   {np.mean(prec_scores):.4f} ± {np.std(prec_scores):.4f}")
print(f"  Recall:      {np.mean(rec_scores):.4f} ± {np.std(rec_scores):.4f}")

print("\nF1 pro Entität (Mittelwert ± Standardabweichung):")
for e in entities:
    scores = [r[f"test_{e}_f1"] for r in seed_results]
    print(f"  {e:12s} {np.mean(scores):.4f} ± {np.std(scores):.4f}")

In [ ]:
import numpy as np

print("Test-Ergebnisse pro Seed:")
for r in seed_results:
    print(f"  Seed {r['seed']}: F1={r['test_f1']:.4f}  Precision={r['test_precision']:.4f}  Recall={r['test_recall']:.4f}")

f1_scores = [r["test_f1"] for r in seed_results]
prec_scores = [r["test_precision"] for r in seed_results]
rec_scores = [r["test_recall"] for r in seed_results]

print("\nÜber alle Seeds gemittelt (Mittelwert ± Standardabweichung):")
print(f"  F1:        {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  Precision: {np.mean(prec_scores):.4f} ± {np.std(prec_scores):.4f}")
print(f"  Recall:    {np.mean(rec_scores):.4f} ± {np.std(rec_scores):.4f}")